# 1. Data Preparation
This [dataset](https://www.kaggle.com/datasets/rabieelkharoua/air-quality-and-health-impact-dataset) contains comprehensive information on the air quality and its impact on public health for 5,811 records. It includes variables such as air quality index (AQI), concentrations of various pollutants, weather conditions, and health impact metrics. The target variable is the health impact class, which categorizes the health impact based on the air quality and other related factors.

This dataset offers a comprehensive view of the relationship between air quality and public health, making it ideal for research, predictive modeling, and statistical analysis.

In [23]:
include("utils.jl")

   Resolving package versions...
  No Changes to `C:\Users\alons\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\alons\.julia\environments\v1.11\Manifest.toml`


averageAccuracies (generic function with 1 method)

In [3]:
using CSV, DataFrames

# Load the dataset from the 'dataset' folder
data = CSV.read("datasets/air_quality_health_impact_data.csv", DataFrame)

# Check the dataset
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,RecordID,2906.0,1,2906.0,5811,0,Int64
2,AQI,248.438,0.00581738,249.128,499.859,0,Float64
3,PM10,148.655,0.0158481,147.635,299.902,0,Float64
4,PM2_5,100.224,0.0315489,100.506,199.985,0,Float64
5,NO2,102.293,0.00962478,102.988,199.98,0,Float64
6,SO2,49.4568,0.0110232,49.5302,99.9696,0,Float64
7,O3,149.312,0.001661,149.56,299.937,0,Float64
8,Temperature,14.9755,-9.991,14.9424,39.9634,0,Float64
9,Humidity,54.7769,10.0015,54.5439,99.9975,0,Float64


In [4]:
# Count elements in each class
counts = combine(groupby(data, :HealthImpactClass), nrow => :Count)

println("Elements per class:")
println(counts)

Elements per class:
5×2 DataFrame
 Row │ HealthImpactClass  Count 
     │ Float64            Int64 
─────┼──────────────────────────
   1 │               0.0   4808
   2 │               1.0    579
   3 │               2.0    273
   4 │               3.0     95
   5 │               4.0     56


In [5]:
input_data = Matrix(data[!, 1:13]);
output_data = Int.(data[!, 15]);

@assert input_data isa Matrix
@assert output_data isa Vector{Int64}

To split the test from train we use the holdOut function with a fraction of 0.2. This means that 20% of the train data will be used for testing and the remaining 80% will be used for training.

In [6]:
# Split in train and test
(tr_idx, test_idx) = holdOutStratified(output_data, 0.2)

train_input = input_data[tr_idx,:]
train_output = output_data[tr_idx]
test_input = input_data[test_idx,:]
test_output = output_data[test_idx]

train_output = collect(train_output)
test_output = collect(test_output)

println("Train Input Size: ", size(train_input))
println("Train Output Size: ", size(train_output), " Categories:", sort(unique(train_output)))
println("Test Input Size: ", size(test_input))
println("Test Output Size: ", size(test_output), " Categories:", sort(unique(test_output)))

Train Input Size: (4648, 13)
Train Output Size: (4648,) Categories:[0, 1, 2, 3, 4]
Test Input Size: (1163, 13)
Test Output Size: (1163,) Categories:[0, 1, 2, 3, 4]


In [ ]:
columna = train_output[:, 1]

conteo = Dict(c => count(==(c), columna) for c in 0:4)

println("Elements per class:")
println(conteo)

Elements per class:
Dict(5 => 0, 4 => 45, 2 => 218, 3 => 76, 1 => 463)


In [ ]:
columna = test_output[:, 1]

conteo = Dict(c => count(==(c), columna) for c in 0:4)

println("Elements per class:")
println(conteo)

Elements per class:
Dict(5 => 0, 4 => 11, 2 => 55, 3 => 19, 1 => 116)


# 2. Definition of Models and Hyperparameters

In [7]:
## These are the modelsHyperParameters to be used in the models training

modelsHyperParameters = [
    # ANN configurations
    Dict("estimator" => :ANN, "topology" => (64,), "maxEpochs" => 200, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (128,), "maxEpochs" => 500, "learningRate" => 0.005),
    # Dict("estimator" => :ANN, "topology" => (64, 32), "maxEpochs" => 400, "learningRate" => 0.001),
    # Dict("estimator" => :ANN, "topology" => (128, 64), "maxEpochs" => 200, "learningRate" => 0.01),
    # Dict("estimator" => :ANN, "topology" => (256,), "maxEpochs" => 300, "learningRate" => 0.05),
    # Dict("estimator" => :ANN, "topology" => (128, 64, 32), "maxEpochs" => 200, "learningRate" => 0.01),
    # Dict("estimator" => :ANN, "topology" => (64, 64), "maxEpochs" => 250, "learningRate" => 0.005),
    # Dict("estimator" => :ANN, "topology" => (256, 128), "maxEpochs" => 200, "learningRate" => 0.001),

    # SVM configurations
    Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 0.5, "degree"=>2),
    Dict("estimator" => :SVM, "kernel" => "linear", "C" => 1.0, "degree"=>4),
    # Dict("estimator" => :SVM, "kernel" => "poly", "C"=> 0.01 , "degree" => 2),
    # Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 1.0, "degree" => 3),
    # Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 0.01, "degree" => 3),
    # Dict("estimator" => :SVM, "kernel" => "poly", "C"=> 5, "degree" => 3),
    # Dict("estimator" => :SVM, "kernel" => "linear", "C"=> 0.5, "degree" => 3),
    # Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 10.0, "degree" => 5),

    # Decision Tree configurations
    Dict("estimator" => :DecisionTree, "max_depth" => 3, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 5, "random_state" => 42),
    # Dict("estimator" => :DecisionTree, "max_depth" => 7, "random_state" => 42),
    # Dict("estimator" => :DecisionTree, "max_depth" => 10, "random_state" => 42),
    # Dict("estimator" => :DecisionTree, "max_depth" => 15, "random_state" => 42),
    # Dict("estimator" => :DecisionTree, "max_depth" => 20, "random_state" => 42),

    # kNN configurations
    Dict("estimator" => :KNN, "k" => 3),
    Dict("estimator" => :KNN, "k" => 5),
    # Dict("estimator" => :KNN, "k" => 7),
    # Dict("estimator" => :KNN, "k" => 9),
    # Dict("estimator" => :KNN, "k" => 11),
    # Dict("estimator" => :KNN, "k" => 15)
]

# println(modelsHyperParameters)

8-element Vector{Dict{String, Any}}:
 Dict("maxEpochs" => 200, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (64,))
 Dict("maxEpochs" => 500, "learningRate" => 0.005, "estimator" => :ANN, "topology" => (128,))
 Dict("estimator" => :SVM, "C" => 0.5, "kernel" => "rbf", "degree" => 2)
 Dict("estimator" => :SVM, "C" => 1.0, "kernel" => "linear", "degree" => 4)
 Dict("estimator" => :DecisionTree, "max_depth" => 3, "random_state" => 42)
 Dict("estimator" => :DecisionTree, "max_depth" => 5, "random_state" => 42)
 Dict("estimator" => :KNN, "k" => 3)
 Dict("estimator" => :KNN, "k" => 5)

----
----

# 3. First approach: Using Min-Max Normalization.

In this approach we will use the Min-Max Normalization technique to normalize the data. And then, realize all the models training to finally compare the results.

We start with the normalization of the data. We use a function that normalize the train_input and test_input with the normalization parameters calculated on train_input.

In [8]:
train_input_minmax, test_input_minmax = normalizeData(train_input, test_input, :MinMax)

([0.3721170395869191 0.3524669235489917 … 0.21428571428571427 0.08333333333333333; 0.10585197934595525 0.9211315429581742 … 0.2857142857142857 0.0; … ; 0.561617900172117 0.01811773220554195 … 0.2857142857142857 0.4166666666666667; 0.6581755593803786 0.20910920900178873 … 0.2857142857142857 0.25], [0.6549053356282272 0.9699971398269888 … 0.2857142857142857 0.16666666666666666; 0.8018932874354561 0.7205841878176384 … 0.5714285714285714 0.16666666666666666; … ; 0.28674698795180725 0.20856018826133502 … 0.35714285714285715 0.25; 0.22306368330464715 0.16752019198546628 … 0.2857142857142857 0.25])

In this approach we wont use crossValidation, so we will train the models on the whole dataset and then evaluate them on the test set.

Pre-normalization data characteristics:

In [56]:
describe(DataFrame(train_input, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,2915.06,1.0,2920.5,5810.0,0,Float64
2,AQI,248.224,0.00581738,248.207,499.859,0,Float64
3,PM10,148.69,0.0158481,147.268,299.902,0,Float64
4,PM2_5,100.403,0.0619464,100.605,199.985,0,Float64
5,NO2,102.042,0.101998,102.237,199.98,0,Float64
6,SO2,49.5373,0.0125139,49.6828,99.9696,0,Float64
7,O3,148.843,0.001661,148.542,299.937,0,Float64
8,Temperature,15.0306,-9.991,15.0582,39.9634,0,Float64
9,Humidity,54.6577,10.0084,54.3833,99.9975,0,Float64


Post-normalization data with min-max:

In [57]:
describe(DataFrame(train_input_minmax, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,0.501646,0.0,0.502582,1.0,0,Float64
2,AQI,0.496582,0.0,0.496548,1.0,0,Float64
3,PM10,0.495769,0.0,0.491026,1.0,0,Float64
4,PM2_5,0.501896,0.0,0.502908,1.0,0,Float64
5,NO2,0.510013,0.0,0.510989,1.0,0,Float64
6,SO2,0.49546,0.0,0.496916,1.0,0,Float64
7,O3,0.496245,0.0,0.495241,1.0,0,Float64
8,Temperature,0.500889,0.0,0.501441,1.0,0,Float64
9,Humidity,0.496163,0.0,0.493115,1.0,0,Float64


In [58]:
accuracies = []
models = []
for (idx, modelHyperParameters) in enumerate(modelsHyperParameters)
    model = deepcopy(genModel(modelHyperParameters))
    fit!(model, train_input_minmax, train_output)
    acc = score(model, test_input_minmax, test_output)
    push!(accuracies, (idx, modelHyperParameters["estimator"], acc))
    push!(models, deepcopy(model))
end;

C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Once we have the accuracies, we will take the best of each model to create the ensemble, in this case we will create a stack, and then train the stack.

In [59]:
println(accuracies)
best_models_position = best_model_positions(accuracies)

stacking_classifier = StackingClassifier(
	estimators = [("m$(idx)_" * string(modelsHyperParameters[idx]["estimator"]), models[idx]) for idx in best_models_position],
	final_estimator = SVC(probability = true), n_jobs = -1)

fit!(stacking_classifier, train_input_minmax, train_output)

Any[(1, :ANN, 0.9355116079105761), (2, :ANN, 0.9294926913155632), (3, :ANN, 0.9509888220120378), (4, :ANN, 0.9337919174548581), (5, :ANN, 0.9492691315563199), (6, :ANN, 0.9303525365434222), (7, :ANN, 0.943250214961307), (8, :ANN, 0.9509888220120378), (9, :SVM, 0.8925193465176269), (10, :SVM, 0.9062768701633706), (11, :SVM, 0.827171109200344), (12, :SVM, 0.8443680137575237), (13, :SVM, 0.827171109200344), (14, :SVM, 0.884780739466896), (15, :SVM, 0.9045571797076526), (16, :SVM, 0.8374892519346517), (17, :DecisionTree, 0.8460877042132416), (18, :DecisionTree, 0.88134135855546), (19, :DecisionTree, 0.8779019776440241), (20, :DecisionTree, 0.8787618228718831), (21, :DecisionTree, 0.883061049011178), (22, :DecisionTree, 0.880481513327601), (23, :KNN, 0.8383490971625107), (24, :KNN, 0.8383490971625107), (25, :KNN, 0.8486672398968186), (26, :KNN, 0.8409286328460877), (27, :KNN, 0.8366294067067928), (28, :KNN, 0.8349097162510748)]


C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(


PyObject StackingClassifier(estimators=[('m25_KNN', KNeighborsClassifier(n_neighbors=7)),
                               ('m10_SVM', SVC(degree=4, kernel='linear')),
                               ('m3_ANN',
                                MLPClassifier(hidden_layer_sizes=(64, 32),
                                              max_iter=400)),
                               ('m21_DecisionTree',
                                DecisionTreeClassifier(max_depth=15,
                                                       random_state=42))],
                   final_estimator=SVC(probability=True), n_jobs=-1)

Now we get the metrics of the stack ensemble.

In [60]:
outputs = stacking_classifier.predict(test_input_minmax)
categories = sort(unique(output_data))

x = copy(outputs)
y = vec(copy(test_output))
metrics = confusionMatrix(x, y)

println(metrics)


Dict{String, Any}("sensitivity" => 2.8497517614577506, "positive_predictive_value" => 4.626951155512693, "specificity" => 4.8245816035830495, "nombre" => "primero", "negative_predictive_value" => 4.897847085813965, "f_score" => 2.9302301071823362, "confusion_matrix" => [951 9 2 0 0; 6 109 1 0 0; 9 4 42 0 0; 5 3 8 3 0; 10 1 0 0 0], "accuracy" => 0.9501289767841788, "error_rate" => 0.04987102321582115)


And print the metrics for this First Approach:

In [61]:
# Display results
println("METRICS 1st APPROACH (Min-Max Normalization):")
println("---------------------")
println("Accuracy: ", metrics["accuracy"])
println("Error Rate: ", metrics["error_rate"])
println("Sensitivity (Recall): ", metrics["sensitivity"])
println("Specificity: ", metrics["specificity"])
println("Precision: ", metrics["positive_predictive_value"])
println("Negative Predictive Value: ", metrics["negative_predictive_value"])
println("F-Score: ", metrics["f_score"])
println("----------------------")
println()

print_confusion_matrix(metrics["confusion_matrix"], ["Class 0", "Class 1", "Class 2", "Class 3", "Class 4"])

METRICS 1st APPROACH (Min-Max Normalization):
---------------------
Accuracy: 0.9501289767841788
Error Rate: 0.04987102321582115
Sensitivity (Recall): 2.8497517614577506
Specificity: 4.8245816035830495
Precision: 4.626951155512693
Negative Predictive Value: 4.897847085813965
F-Score: 2.9302301071823362
----------------------

CONFUSION MATRIX:
             Class 0   Class 1   Class 2   Class 3   Class 4
   Class 0       951         9         2         0         0
   Class 1         6       109         1         0         0
   Class 2         9         4        42         0         0
   Class 3         5         3         8         3         0
   Class 4        10         1         0         0         0


## In this point we should make the analisys of the results

----
----

# 4. Second Approach: Using Standard normalization

In this approach we will use the Zero-Mean Normalization technique to normalize the data. And then, train the models to finally compare the results.

We start with the normalization of the data. We use a function that normalize the train_input and test_input with the normalization parameters calculated on train_input.

In [62]:
train_input_zm, test_input_zm = normalizeData(train_input, test_input, :MinMax)

([0.8571182647615768 0.03926495927841305 … 0.21428571428571427 0.25; 0.36099156481322087 0.6347573770060905 … 0.21428571428571427 0.25; … ; 0.2587364434498192 0.3413365326594346 … 0.21428571428571427 0.08333333333333333; 0.8385264245136856 0.1322959702609074 … 0.07142857142857142 0.25], [0.08400757445343432 0.9506793124086401 … 0.14285714285714285 0.16666666666666666; 0.040282320537097606 0.19528876041361473 … 0.21428571428571427 0.4166666666666667; … ; 0.9908762265450164 0.6308090937957337 … 0.42857142857142855 0.25; 0.38250989843346533 0.09655121777790915 … 0.5 0.08333333333333333])

Post-normalization data with zero-mean:

In [63]:
describe(DataFrame(train_input_zm, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,0.501646,0.0,0.502582,1.0,0,Float64
2,AQI,0.496582,0.0,0.496548,1.0,0,Float64
3,PM10,0.495769,0.0,0.491026,1.0,0,Float64
4,PM2_5,0.501896,0.0,0.502908,1.0,0,Float64
5,NO2,0.510013,0.0,0.510989,1.0,0,Float64
6,SO2,0.49546,0.0,0.496916,1.0,0,Float64
7,O3,0.496245,0.0,0.495241,1.0,0,Float64
8,Temperature,0.500889,0.0,0.501441,1.0,0,Float64
9,Humidity,0.496163,0.0,0.493115,1.0,0,Float64


In [64]:
accuracies = []
models = []
for (idx, modelHyperParameters) in enumerate(modelsHyperParameters)
    model = genModel(modelHyperParameters)
    fit!(model, train_input_zm, train_output)
    acc = score(model, test_input_zm, test_output)
    push!(accuracies, (idx, modelHyperParameters["estimator"], acc))
    push!(models, deepcopy(model))
end;

C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [65]:
println(accuracies)
best_models_position = best_model_positions(accuracies)

stacking_classifier = StackingClassifier(
	estimators = [("m$(idx)_" * string(modelsHyperParameters[idx]["estimator"]), models[idx]) for idx in best_models_position],
	final_estimator = SVC(probability = true), n_jobs = -1)

fit!(stacking_classifier, train_input_zm, train_output)

Any[(1, :ANN, 0.944969905417025), (2, :ANN, 0.939810834049871), (3, :ANN, 0.9501289767841788), (4, :ANN, 0.9415305245055889), (5, :ANN, 0.9509888220120378), (6, :ANN, 0.9346517626827171), (7, :ANN, 0.945829750644884), (8, :ANN, 0.938091143594153), (9, :SVM, 0.8925193465176269), (10, :SVM, 0.9062768701633706), (11, :SVM, 0.827171109200344), (12, :SVM, 0.8443680137575237), (13, :SVM, 0.827171109200344), (14, :SVM, 0.884780739466896), (15, :SVM, 0.9045571797076526), (16, :SVM, 0.8374892519346517), (17, :DecisionTree, 0.8460877042132416), (18, :DecisionTree, 0.88134135855546), (19, :DecisionTree, 0.8779019776440241), (20, :DecisionTree, 0.8787618228718831), (21, :DecisionTree, 0.883061049011178), (22, :DecisionTree, 0.880481513327601), (23, :KNN, 0.8383490971625107), (24, :KNN, 0.8383490971625107), (25, :KNN, 0.8486672398968186), (26, :KNN, 0.8409286328460877), (27, :KNN, 0.8366294067067928), (28, :KNN, 0.8349097162510748)]


C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


PyObject StackingClassifier(estimators=[('m25_KNN', KNeighborsClassifier(n_neighbors=7)),
                               ('m10_SVM', SVC(degree=4, kernel='linear')),
                               ('m5_ANN',
                                MLPClassifier(hidden_layer_sizes=(256,),
                                              learning_rate_init=0.05,
                                              max_iter=300)),
                               ('m21_DecisionTree',
                                DecisionTreeClassifier(max_depth=15,
                                                       random_state=42))],
                   final_estimator=SVC(probability=True), n_jobs=-1)

In [67]:
outputs = stacking_classifier.predict(test_input_zm)
categories = sort(unique(output_data))

x = copy(outputs)
y = vec(copy(test_output))

metrics = confusionMatrix(x, y)

Dict{String, Any} with 9 entries:
  "sensitivity"               => 2.64748
  "positive_predictive_value" => 4.46849
  "specificity"               => 4.78496
  "nombre"                    => "primero"
  "negative_predictive_value" => 4.86689
  "f_score"                   => 2.62305
  "confusion_matrix"          => [948 12 … 0 0; 8 96 … 0 0; … ; 6 1 … 1 0; 11 0…
  "accuracy"                  => 0.935512
  "error_rate"                => 0.0644884

In [102]:
# Display results
println("METRICS 1st APPROACH (Min-Max Normalization):")
println("---------------------")
println("Accuracy: ", metrics["accuracy"])
println("Error Rate: ", metrics["error_rate"])
println("Sensitivity (Recall): ", metrics["sensitivity"])
println("Specificity: ", metrics["specificity"])
println("Precision: ", metrics["positive_predictive_value"])
println("Negative Predictive Value: ", metrics["negative_predictive_value"])
println("F-Score: ", metrics["f_score"])
println("----------------------")
println()

print_confusion_matrix(metrics["confusion_matrix"], ["Class 0", "Class 1", "Class 2", "Class 3", "Class 4"])

METRICS 1st APPROACH (Min-Max Normalization):
---------------------
Accuracy: 0.9355116079105761
Error Rate: 0.06448839208942392
Sensitivity (Recall): 2.6474829531090873
Specificity: 4.784960807765914
Precision: 4.468494848050085
Negative Predictive Value: 4.86688662199166
F-Score: 2.6230507832641097
----------------------

CONFUSION MATRIX:
             Class 0   Class 1   Class 2   Class 3   Class 4
   Class 0       948        12         2         0         0
   Class 1         8        96        12         0         0
   Class 2        11         1        43         0         0
   Class 3         6         1        11         1         0
   Class 4        11         0         0         0         0


----
----

# 5. Third Approach: Using Crossvalidation

In this approach we will use the Crossvalidation technique to train the models, and using the normalization of the data that gave us the best results in the previous steps.

In [11]:
kFoldIndices = crossvalidation(size(train_output, 1), 5)
# kFoldIndices = reshape(kFoldIndices, 1)

4648-element Vector{Int64}:
 4
 3
 4
 3
 4
 1
 5
 1
 2
 5
 ⋮
 2
 4
 4
 3
 4
 1
 5
 5
 1

In [22]:

metrics_given = trainClassEnsemble(modelsHyperParameters,
    (train_input_minmax, train_output),
    kFoldIndices
    )

C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\conda\3\x86_64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\alons\.julia\co

Dict{String, Any} with 8 entries:
  "sensitivity"               => (2.50878, 0.110485)
  "positive_predictive_value" => (4.61232, 0.0583476)
  "specificity"               => (4.72913, 0.0519525)
  "negative_predictive_value" => (4.90502, 0.0129368)
  "f_score"                   => (2.55413, 0.0757209)
  "accuracy"                  => (0.936534, 0.00960325)
  "error_rate"                => (0.0634659, 0.00960325)
  "models_accuracies"         => Tuple{Int64, Any, Float64}[(1, :ANN, 0.935457)…

### At this point we have the accuracies of the models and the metrics of the stack. We can compare directly the results with the Approach 1 (MinMax).

____
____